# 🌿 AgroGuard Foliar Disease Model Training Pipeline
### MobileNetV2 Transfer Learning for Corn Disease Diagnosis (PlantVillage Benchmark)

This notebook trains a production-grade **MobileNetV2** deep learning model for the **AgroGuard** agronomic platform on **4,188 annotated corn foliar images** across 4 key classes:
1. **Healthy Corn**
2. **Corn Common Rust** (*Puccinia sorghi*)
3. **Northern Leaf Blight** (*Exserohilum turcicum*)
4. **Corn Gray Leaf Spot** (*Cercospora zeae-maydis*)

---
### 📋 Instructions (3 Simple Steps):
1. **Select GPU:** Click `Runtime` ➔ `Change runtime type` ➔ Select **T4 GPU** ➔ Click `Save`.
2. **Run All:** Click `Runtime` ➔ `Run all` (or press `Ctrl + F9`).
3. **Wait ~5 Minutes:** The dataset will download automatically, train for 5 epochs (>97% accuracy), display the training graphs for your project report, and automatically download `disease_model.pt` directly to your computer!

## Step 1: Environment & GPU Verification

In [ ]:
import os
import shutil
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 65)
print(f"PyTorch Version : {torch.__version__}")
print(f"Compute Device  : {device}")
if torch.cuda.is_available():
    print(f"GPU Model       : {torch.cuda.get_device_name(0)}")
    print("✓ Accelerated GPU training active!")
else:
    print("⚠️ Running on CPU. Go to Runtime -> Change runtime type -> T4 GPU for 10x faster training!")
print("=" * 65)

## Step 2: Automatic Dataset Ingestion (Zero Manual Upload Needed)
We use `kagglehub` to download the **4,188-image PlantVillage Corn Dataset** directly into Colab in ~15 seconds without requiring any Kaggle login or local uploads.

In [ ]:
# Ensure kagglehub is installed
!pip install -q kagglehub

import kagglehub

print("📥 Fetching PlantVillage Corn Pathology Dataset...")
raw_dir = kagglehub.dataset_download("smaranjitghose/corn-or-maize-leaf-disease-dataset")
print(f"✓ Dataset downloaded to: {raw_dir}")

# Check subdirectories
subdirs = [d for d in os.listdir(raw_dir) if os.path.isdir(os.path.join(raw_dir, d))]
data_dir = os.path.join(raw_dir, "data") if "data" in subdirs else raw_dir

print("\n📁 Class Distributions:")
total_imgs = 0
for cls_name in sorted(os.listdir(data_dir)):
    cls_path = os.path.join(data_dir, cls_name)
    if os.path.isdir(cls_path):
        imgs = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        total_imgs += len(imgs)
        print(f"  • {cls_name:<20}: {len(imgs):>4} images")
print(f"✓ Total Annotated Images: {total_imgs}")

## Step 3: Data Preprocessing & Augmentation
Agronomic data augmentations (flips, rotation, color jitter) simulate variable sunlight, field shadows, and leaf orientations.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_dataset = datasets.ImageFolder(data_dir, transform=train_transform)
raw_classes = full_dataset.classes

# 80% Train, 20% Validation split
train_size = int(0.80 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset_raw = random_split(
    full_dataset, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

# Validation dataset with clean deterministic transform
val_dataset_clean = datasets.ImageFolder(data_dir, transform=val_transform)
val_dataset = torch.utils.data.Subset(val_dataset_clean, val_dataset_raw.indices)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Training set   : {len(train_dataset)} images (80%)")
print(f"Validation set : {len(val_dataset)} images (20%)")
print(f"Target classes : {raw_classes}")

## Step 4: MobileNetV2 Architecture Setup

In [ ]:
# Load MobileNetV2 with ImageNet pre-trained weights
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Freeze base feature extractor for transfer learning
for param in model.features.parameters():
    param.requires_grad = False

# Replace classification head for 4 Corn Pathology classes
num_classes = len(raw_classes)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(model.last_channel, num_classes)
)

model = model.to(device)
print(f"✓ MobileNetV2 loaded with custom {num_classes}-class classifier head on {device}.")

## Step 5: Training Loop (5 Epochs — ~3 Minutes on GPU)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

EPOCHS = 5
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_weights = copy.deepcopy(model.state_dict())

start_time = time.time()
print("Starting Model Training...\n")

for epoch in range(EPOCHS):
    # Training Phase
    model.train()
    running_loss, running_corrects = 0.0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        
    train_loss = running_loss / len(train_dataset)
    train_acc = (running_corrects.double() / len(train_dataset)).item() * 100

    # Validation Phase
    model.eval()
    val_loss, val_corrects = 0.0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(val_dataset)
    val_acc = (val_corrects.double() / len(val_dataset)).item() * 100
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_weights = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1}/{EPOCHS}] | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

elapsed = time.time() - start_time
print(f"\n✓ Training Completed in {elapsed/60:.2f} minutes!")
print(f"✓ Highest Validation Accuracy: {best_val_acc:.2f}%")
model.load_state_dict(best_weights)

## Step 6: Evaluation Artifacts for Project Report & PPT
Run this cell to save high-resolution figures for your college documentation:
- `agroguard_training_curves.png` (Epochs vs Loss & Accuracy)
- `agroguard_confusion_matrix.png` (Diagnostic Confusion Matrix)

In [ ]:
# 1. Plot Accuracy & Loss Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS + 1)

ax1.plot(epochs_range, history['train_loss'], 'o-', color='#e74c3c', lw=2.5, label='Training Loss')
ax1.plot(epochs_range, history['val_loss'], 's--', color='#c0392b', lw=2.5, label='Validation Loss')
ax1.set_title('Cross-Entropy Loss vs. Epochs', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=10)
ax1.set_ylabel('Loss', fontsize=10)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend()

ax2.plot(epochs_range, history['train_acc'], 'o-', color='#2ecc71', lw=2.5, label='Training Accuracy')
ax2.plot(epochs_range, history['val_acc'], 's--', color='#27ae60', lw=2.5, label='Validation Accuracy')
ax2.set_title('Diagnostic Accuracy vs. Epochs (%)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=10)
ax2.set_ylabel('Accuracy (%)', fontsize=10)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.savefig('agroguard_training_curves.png', dpi=300)
plt.show()

# 2. Confusion Matrix & Classification Report
all_preds = []
all_labels = []
model.eval()
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

canonical_names = {
    'Blight': 'Northern Leaf Blight',
    'Common_Rust': 'Corn Common Rust',
    'Gray_Leaf_Spot': 'Corn Gray Leaf Spot',
    'Healthy': 'Healthy'
}
display_labels = [canonical_names.get(c, c) for c in raw_classes]

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=display_labels, yticklabels=display_labels)
plt.title('AgroGuard Corn Pathology Confusion Matrix', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Diagnosis', fontsize=10)
plt.ylabel('Ground Truth (Pathologist Labeled)', fontsize=10)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('agroguard_confusion_matrix.png', dpi=300)
plt.show()

print("=" * 65)
print("           DETAILED CLASSIFICATION REPORT")
print("=" * 65)
print(classification_report(all_labels, all_preds, target_names=display_labels))

## Step 7: Export & Automatic Browser Download (`disease_model.pt`)
This step saves the model in the format expected by AgroGuard backend (`backend/app/ml_models/disease_model.pt`) and automatically prompts your browser to download it.

In [ ]:
# Save checkpoint with AgroGuard canonical classes metadata
checkpoint = {
    'state_dict': model.state_dict(),
    'classes': display_labels,
    'raw_classes': raw_classes,
    'accuracy': float(best_val_acc),
    'architecture': 'mobilenet_v2',
    'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
}

save_path = 'disease_model.pt'
torch.save(checkpoint, save_path)
file_size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(f"✓ Model checkpoint saved: {save_path} ({file_size_mb:.2f} MB)")

# Automatic browser download in Colab
try:
    from google.colab import files
    print("\n⬇️ Triggering automatic browser download...")
    files.download(save_path)
    files.download('agroguard_training_curves.png')
    files.download('agroguard_confusion_matrix.png')
    print("✓ Browser download initiated!")
except Exception as e:
    print(f"Note: {e}. You can manually download disease_model.pt from the left file panel.")